## Earthquake modelling
During an earthquake, the ground experiences material strain as a function of location.
Material strain during earthquakes can be measured and stored in 'seismograms'.
This can be translated directly into distributed optical strain in optical fibres, affecting differential group delay.
Synthetic seismograms can be obtained by HTTPS request from [Syngine](https://ds.iris.edu/ds/products/syngine/), which uses a spectral element method and databases with measured material strain.
This notebook demonstrates how to obtain such seismograms (an internet connection is required!), and use them to perturb the waveplate model (see demo_waveplate.ipynb).

As a demonstration, we'll retrieve seismogram data along the Curie cable between Valparaiso and Los Angeles.
The curie cable goes from ~ `-33.05` latitude, `-71.62` longitude to ~ `33.92` latitude, `-118.42` longitude.
We start by retrieving the estimate path from https://www.submarinecablemap.com/api/v3/cable/cable-geo.json.

In [ ]:
from configparser import ConfigParser

# Cable coordinates from https://www.submarinecablemap.com/api/v3/cable/cable-geo.json
parameters = ConfigParser()

parameters['FIBRE'] = {
    'section_length':    '100',  # Correlation length in km
    'path_coordinates':  '[ \
        [-71.62043502747198,-33.04554123247811], \
        [-72.67487428049728,-31.85146566557725], \
        [-79.64986933934534,-18.45381377577717], \
        [-84.14986615150535,-11.503333845984299], \
        [-85.4998651951533,-6.616650693475355], \
        [-85.94986487636929,0.568578852526193], \
        [-87.06475643430802,5.534179292238662], \
        [-92.69986009460932,9.52441134501949], \
        [-98.09985626920157,12.615395567393394], \
        [-104.39985180622544,16.10232559580297], \
        [-108.44984893716946,18.678647022154717], \
        [-114.29984479297735,22.469443964829516], \
        [-118.79984160513736,27.76358852605777], \
        [-119.6998409675696,31.286738814391754], \
        [-119.24984128635361,32.8123187832876], \
        [-118.79984160513764,33.75276987113061], \
        [-118.41596126184768,33.91992001851462] \
    ]',                          # Estimate coordinates of the Curie cable, taken from https://www.submarinecablemap.com/api/v3/cable/cable-geo.json
    'PMD_parameter':     '0.1',  # Polarisation mode dispersion parameter in ps / (km ^ 0.5)
    'realisation_count': '1000', # Number of fibre realisations to simulate simultaneously
    'photoelasticity':   '0.1'   # Photoelasticity, which relates material strain to optical strain
}

parameters['EARTHQUAKE'] = {
    'event': 'GCMT:C201002270634A', # A historic earthquake event, structured <catalog>:<identifier> (e.g. from https://www.globalcmt.org/)
    'model': 'ak135f_5s'             # Earth model for Syngine to use from https://ds.iris.edu/ds/products/syngine/#earth
}

The Fibre class of the toolbox automatically infers PMD section coordinates using linear interpolation.
See demo_fibre.ipynb for a more detailed example of this class.

In [ ]:
from tremor_waveplate_toolbox import Fibre
curie = Fibre(parameters)

Let's plot the obtained fibre path, and interpolated sections:

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw = {'projection': ccrs.LambertCylindrical()})

ax.add_feature(cfeature.LAND, facecolor = 'forestgreen')
ax.add_feature(cfeature.OCEAN, facecolor = 'paleturquoise')
ax.add_feature(cfeature.LAKES, facecolor = 'paleturquoise', edgecolor = 'black')
ax.add_feature(cfeature.COASTLINE)
ax.set_xlim([-130, -50])
ax.set_ylim([-50, 50])

ax.plot(curie.section_coordinates[:,0], curie.section_coordinates[:,1], 'o')
ax.plot(curie.path_coordinates[:,0], curie.path_coordinates[:,1], 'o')
ax.text(curie.path_coordinates[0,0], curie.path_coordinates[0,1], "Valparaiso", color = 'whitesmoke')
ax.text(curie.path_coordinates[-1,0], curie.path_coordinates[-1,1], "Los Angeles", color = 'whitesmoke')
ax.legend(["Interpolated coordinates", "Estimate route"], loc = 'lower left')

plt.savefig("map.pdf", bbox_inches = 'tight')

Now, we use the Earthquake class to load seismogram data at the fibre section coordinates.
Internally, it requests seismograms from Syngine by HTTP request.

We obtain seismograms at every fibre section end (so if we have S sections, we obtain S + 1 seismograms).
The obtained seismograms are measured along four axes:  
1. The normal axis, facing away from the earth's core.  
2. The longitudinal axis, measuring ground displacement from west to east.  
3. The latitudinal axis, measuring ground displacement from south to north.  
4. The projected axis, measuring ground displacement in each section's direction.

Additionally, we obtain fibre material strain along the projected axis.

Note: Syngine assumes receivers to be at 0 depth (sea floor- or surface level).
Therefore, the synthetic seismograms may not correspond exactly with the actual tremors experienced by the fibre.
This could be improved by obtaining ocean floor depths using e.g. [oceansdb](https://pypi.org/project/oceansdb/), and obtaining seismograms using e.g. [Pyrocko-GF](https://pyrocko.org/docs/current/topics/pyrocko-gf.html).
However, Pyrocko-GF has some downsides:
- it has no online interface like Syngine,
- it provides much less detailed datasets by default.

In [ ]:
from tremor_waveplate_toolbox import Earthquake

# Model an earthquake from the GCMT database (https://www.globalcmt.org/). This ID represents an earthquake in Chile on 2010/02/27 with a moment magnitude of 8.8
# We use the ak135f_1s earth model to generate synthetic waveforms, see https://ds.iris.edu/ds/products/syngine/
earthquake = Earthquake(parameters)

times, seismograms_normal, seismograms_longitude, seismograms_latitude, seismograms_projected, strains_projected = \
    earthquake(curie, verbose = True)

Let's plot the obtained seismograms as surface plots, and save the data as .csv for plotting in other software:

In [ ]:
import csv
import numpy as np

# Make some surface plots
fig, axes = plt.subplots(6, figsize = (12, 35), subplot_kw={'projection': '3d'})

def plot_seismograms(ax, seismograms, title, vmin = -.02, vmax = .02):
    im = ax.plot_surface(
        times[None] / 60,
        curie.section_positions[:,None],
        seismograms,
        cmap = 'seismic',
        vmin = vmin,
        vmax = vmax
    )
    ax.set_xlabel("Time [min]")
    ax.set_ylabel("Fibre position [km]")
    ax.set_title(title)
    fig.colorbar(im, ax = ax)

plot_seismograms(axes[0], seismograms_normal, "Normal displacement [m]")
plot_seismograms(axes[1], seismograms_longitude, "Longitudinal displacement [m]")
plot_seismograms(axes[2], seismograms_latitude, "Latitudinal displacement [m]")


def plot_projected_seismograms(ax, seismograms, title, vmin = -.02, vmax = .02):
    im = ax.plot_surface(
        times[None] / 60,
        curie.section_centre_positions[:,None],
        seismograms,
        cmap = 'seismic',
        vmin = vmin,
        vmax = vmax
    )
    ax.set_xlabel("Time [min]")
    ax.set_ylabel("Fibre position [km]")
    ax.set_title(title)
    fig.colorbar(im, ax = ax)

plot_projected_seismograms(axes[3], seismograms_projected[:, :, 0], "Projected displacement (section starts) [m]")
plot_projected_seismograms(axes[4], seismograms_projected[:, :, 1], "Projected displacement (section ends) [m]")
plot_projected_seismograms(axes[5], strains_projected, "Projected material strain", vmin = -4e-5, vmax = 4e-5)

# fig.savefig(f"seismograms_{np.diff(curie.section_positions)[0]:.1f}m_{np.diff(times)[0]:.2f}s.png")

# Save data .csv to animate using other software
step = 100 # Save 1 in 100 frames

with open(f"seismograms_world_{np.diff(curie.section_positions)[0]:.1f}m_{np.diff(times)[0]:.2f}s" + ".csv", 'w', newline = '') as csv_file:
    seismogram_writer = csv.writer(csv_file)
    seismogram_writer.writerow(
        ["Distance"] +
        [f"Normal{frame}"    for frame in range(int(len(times) / step))] +
        [f"Longitude{frame}" for frame in range(int(len(times) / step))] +
        [f"Latitude{frame}"  for frame in range(int(len(times) / step))]
    ) # Table header

    for section_end_index, section_end_position in enumerate(curie.section_positions):
        seismogram_writer.writerow(
            [section_end_position] +
            seismograms_normal[section_end_index, :-step:step].tolist() +
            seismograms_longitude[section_end_index, :-step:step].tolist() +
            seismograms_latitude[section_end_index, :-step:step].tolist()
        )

with open(f"seismograms_projected_{np.diff(curie.section_positions)[0]:.1f}m_{np.diff(times)[0]:.2f}s" + ".csv", 'w', newline = '') as csv_file:
    seismogram_writer = csv.writer(csv_file)
    seismogram_writer.writerow(
        ["Distance"] +
        [f"Start{frame}"  for frame in range(int(len(times) / step))] +
        [f"End{frame}"    for frame in range(int(len(times) / step))] +
        [f"Strain{frame}" for frame in range(int(len(times) / step))]
    ) # Table header

    for section_end_index, section_end_position in enumerate(curie.section_centre_positions):
        seismogram_writer.writerow(
            [section_end_position] +
            seismograms_projected[section_end_index, :-step:step, 0].tolist() +
            seismograms_projected[section_end_index, :-step:step, 1].tolist() +
            strains_projected[section_end_index, :-step:step].tolist()
        )